# K230D YOLOv8n — end-to-end (Docker, no venvs)

Readable one-notebook flow for **convert → simulate → compare → deploy**.
All heavy logic lives in `k230_pipeline.py` (same folder, mounted at `/workspace`);
the cells below are thin calls so you can read the whole pipeline at a glance.

### How to launch (from the `k230-final-train` folder on the host)
```
docker compose up lab        # -> http://localhost:8888/lab  then open this file
```

### Training happens first, in the GPU container (different image)
```
docker compose run --rm train python k230_pipeline.py train \
    --data datasets/victim_20260621/data.yaml --name victim
docker compose run --rm train python k230_pipeline.py export \
    runs/victim/weights/best.pt --output models
```
That produces `models/best_640x480.onnx`, which this notebook consumes.

## 0. Setup — config in ONE place
Edit these, run once. (`opencv-python-headless` is added because the nncase
lab image doesn't ship it; everything else is already in the image.)

In [ ]:
import sys, subprocess
try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'opencv-python-headless==4.10.0.84'])

import k230_pipeline as P

# ---- CONFIG -------------------------------------------------------------
ONNX        = 'models/best_640x480.onnx'                 # from the train container
CALIB_DIR   = 'datasets/victim_20260621/val/images'      # PTQ calibration images
EVAL_DIR    = 'datasets/victim_20260621/val/images'      # images used to simulate/score
NUM_CLASSES = 3                                          # Dead / Live / Point
IMG_W, IMG_H = 640, 480
NUM_SAMPLES = 8                                          # calibration images
EVAL_LIMIT  = 20                                         # images scored in sim
print('config OK — nncase will compile for target k230')

## 1. Convert — MODE A: SEARCH
Sweep PTQ options `[0,3,1,4]`, simulate each kmodel against the float ONNX
(cosine similarity + detection-confidence delta), rank, and save the winner to
`best_model/searched/` with full `params.json`.

In [ ]:
P.api_convert_search(ONNX, CALIB_DIR, eval_data=EVAL_DIR,
                     num_classes=NUM_CLASSES, img_width=IMG_W, img_height=IMG_H,
                     num_samples=NUM_SAMPLES, eval_limit=EVAL_LIMIT)

## 2. Convert — MODE B: PRESET
Compile the ONE predefined verified-best config (`ptq_option 0`: NoClip,
uint8 act + uint8 weights, Canaan `[0,1]`/`std[1,1,1]`) to `best_model/preset/`.

In [ ]:
P.api_convert_preset(ONNX, CALIB_DIR, eval_data=EVAL_DIR,
                     num_classes=NUM_CLASSES, img_width=IMG_W, img_height=IMG_H,
                     num_samples=NUM_SAMPLES, eval_limit=EVAL_LIMIT)

## 3. Compare — searched vs preset (speed + accuracy)
Writes `best_model/comparison.json` with a per-metric verdict.

In [ ]:
P.api_compare()

## 4. Deploy — bundle the chosen model
Copies the kmodel + labels + `deploy_config.json` + the CanMV on-device script
into `deploy/`. Change `which='preset'` to ship the preset instead.

In [ ]:
P.api_deploy(which='searched', data='data.yaml',
             img_width=IMG_W, img_height=IMG_H, conf=0.30, iou=0.45)

## Done
Copy the 4 files in `deploy/` to `/data/k230-final-train/` on the K230D SD card
(+ a `test.jpg`), then from the CanMV REPL:
```python
import deploy_canmv_yolov8; deploy_canmv_yolov8.detection()
```